# Public portfolio adaptation
Adapted from my Fall 2025 Gilead project. Outputs and original data are excluded.
Company examples are fictional. Requires your own Databricks workspace and credentials.
This notebook is retained as research code; see the repository documentation for limitations.


# Window Size Testing: Claude, FinBERT, and Llama

This notebook tests different window sizes for sentiment analysis across three models:
- **Claude**: Anthropic's Claude model via API
- **FinBERT**: Financial domain-specific BERT model
- **Llama**: Meta's Llama model via Databricks endpoint

**Data Limit**: Testing on first 100 sentences only (for faster iteration)

**Execution Order**: Llama → FinBERT → Claude (Llama runs first to catch errors early)

**Window Sizes Tested**:
- **1 chunk**: [n] - Current sentence only
- **3 chunks**: [n-1, n, n+1] - Previous, current, and next sentence
- **5 chunks**: [n-2, n-1, n, n+1, n+2] - Two before, current, two after
- **7 chunks**: [n-3, n-2, n-1, n, n+1, n+2, n+3] - Three before, current, three after

**Score Values**:
- **NA (null)**: Only at boundaries where context is missing
- **0.0**: Only when model explicitly returns 0.0 for objective (non-commentary) facts
- **"Error"**: When model does not produce a valid response (timeout, parsing errors, invalid scores, missing responses, etc.)

In [ ]:
# Install required packages
%pip install anthropic transformers torch accelerate mlflow python-dotenv
%restart_python

In [ ]:
# ==================================================
# IMPORTS AND SETUP
# ==================================================
import anthropic
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
import json
import re
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from mlflow.deployments import get_deploy_client
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import sleep
import os
from dotenv import load_dotenv

spark = SparkSession.builder.getOrCreate()

print("✅ All imports successful")

In [ ]:
# ==================================================
# MODEL SETUP
# ==================================================

# Claude setup
# Load environment variables from .env file
# Create a .env file in the Databricks folder with: CLAUDE_API_KEY=your-api-key-here
load_dotenv()

api_key = os.getenv("CLAUDE_API_KEY")
if not api_key:
    raise ValueError("CLAUDE_API_KEY not found in environment. Please create a .env file with CLAUDE_API_KEY=your-api-key-here")

claude_client = anthropic.Anthropic(api_key=api_key)

# FinBERT setup
FINBERT_MODEL = "ProsusAI/finbert"

# Databricks endpoints
db_client = get_deploy_client("databricks")
LLAMA = "databricks-meta-llama-3-1-8b-instruct"

print("✅ Model configurations set")

In [ ]:
# ==================================================
# HELPER FUNCTIONS
# ==================================================
import math

def polarity_to_1_7(p_pos: float, p_neg: float) -> float:
    """Convert polarity scores to 1-7 scale"""
    return float(max(1.0, min(7.0, 4.0 + 3.0 * (p_pos - p_neg))))

def round_to_sig_figs(value: float, sig_figs: int = 3) -> float:
    """Round a number to specified significant figures (e.g., 5.45 for 3 sig figs)"""
    if value == 0.0:
        return 0.0
    import math
    return round(value, sig_figs - int(math.floor(math.log10(abs(value)))) - 1)



def extract_json(text):
    """Robust JSON extraction from model responses - ALWAYS returns a dict with 'score' key - never returns None"""
    # Handle None or non-string input
    if text is None or not isinstance(text, str):
        return {"score": 4}
    
    # Try to parse as complete JSON
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict) and "score" in parsed:
            return parsed
    except:
        pass
    
    # Try to find JSON object with score
    match = re.search(r'\{[^}]*"score"[^}]*\}', text)
    if match:
        try:
            parsed = json.loads(match.group())
            if isinstance(parsed, dict):
                return parsed
        except:
            pass
    
    # Try to extract just the score number
    number_match = re.search(r'"score":\s*([0-9.]+)', text)
    if number_match:
        try:
            score_val = float(number_match.group(1))
            return {"score": score_val}
        except:
            pass
    
    # ALWAYS return a dict - never None (matches working version)
    return {"score": 4}

print("✅ Helper functions defined")

In [ ]:
# ==================================================
# LOAD & CLEAN DATA (LIMITED TO FIRST 100 ROWS)
# ==================================================
print("Loading data from parsed_data table...")
df = spark.table("parsed_data")

src = (df
    .withColumn("row_idx", F.monotonically_increasing_id())
    .withColumn("report_id", F.col("`Doc #`").cast("long"))
    .withColumn("sentence_number", F.col("`Sentence #`").cast("long"))
    .withColumn("sentence", F.substring(F.col("Context"), 1, 2000))
    .filter(F.col("report_id").isNotNull())
    .select("report_id", "row_idx", "sentence_number", "sentence")
    .orderBy("report_id", "row_idx")
    .limit(100)  # Limit to first 100 rows for faster testing
)

# Create window to access previous and next sentences
w = Window.partitionBy("report_id").orderBy("row_idx")

windowed = (src
    .withColumn("sentence_n_minus3", F.lag("sentence", 3).over(w))
    .withColumn("sentence_n_minus2", F.lag("sentence", 2).over(w))
    .withColumn("sentence_n_minus1", F.lag("sentence", 1).over(w))
    .withColumn("sentence_n", F.col("sentence"))
    .withColumn("sentence_n_plus1", F.lead("sentence", 1).over(w))
    .withColumn("sentence_n_plus2", F.lead("sentence", 2).over(w))
    .withColumn("sentence_n_plus3", F.lead("sentence", 3).over(w))
)

print(f"✅ Data loaded: {src.count()} sentences (limited to first 100 for testing)")
display(windowed.limit(5))



In [ ]:
# ==================================================
# CREATE 4 DIFFERENT WINDOW CONFIGURATIONS
# ==================================================
print("Creating 4 window configurations...")
print("Every sentence will be scored for each window size.")
print("At boundaries, scores will be NA if context is missing.\n")

# Window 1: [n] - 1 chunk (current sentence only)
window_1chunk = windowed.select(
    "report_id", "sentence_number",
    F.lit("1_chunk").alias("window_type"),
    F.col("sentence_n").alias("window_text"),
    F.lit(True).alias("has_full_context")
).filter(F.col("sentence_n").isNotNull())

# Window 3: [n-1, n, n+1] - 3 chunks
window_3chunk = windowed.select(
    "report_id", "sentence_number",
    F.lit("3_chunk").alias("window_type"),
    F.when(
        F.col("sentence_n_minus1").isNotNull() & 
        F.col("sentence_n").isNotNull() & 
        F.col("sentence_n_plus1").isNotNull(),
        F.concat_ws(" ", 
            F.col("sentence_n_minus1"),
            F.col("sentence_n"),
            F.col("sentence_n_plus1")
        )
    ).otherwise(F.lit(None)).alias("window_text"),
    (F.col("sentence_n_minus1").isNotNull() & 
     F.col("sentence_n").isNotNull() & 
     F.col("sentence_n_plus1").isNotNull()).alias("has_full_context")
).filter(F.col("sentence_n").isNotNull())

# Window 5: [n-2, n-1, n, n+1, n+2] - 5 chunks
window_5chunk = windowed.select(
    "report_id", "sentence_number",
    F.lit("5_chunk").alias("window_type"),
    F.when(
        F.col("sentence_n_minus2").isNotNull() &
        F.col("sentence_n_minus1").isNotNull() & 
        F.col("sentence_n").isNotNull() & 
        F.col("sentence_n_plus1").isNotNull() &
        F.col("sentence_n_plus2").isNotNull(),
        F.concat_ws(" ",
            F.col("sentence_n_minus2"),
            F.col("sentence_n_minus1"),
            F.col("sentence_n"),
            F.col("sentence_n_plus1"),
            F.col("sentence_n_plus2")
        )
    ).otherwise(F.lit(None)).alias("window_text"),
    (F.col("sentence_n_minus2").isNotNull() &
     F.col("sentence_n_minus1").isNotNull() & 
     F.col("sentence_n").isNotNull() & 
     F.col("sentence_n_plus1").isNotNull() &
     F.col("sentence_n_plus2").isNotNull()).alias("has_full_context")
).filter(F.col("sentence_n").isNotNull())

# Window 7: [n-3, n-2, n-1, n, n+1, n+2, n+3] - 7 chunks
window_7chunk = windowed.select(
    "report_id", "sentence_number",
    F.lit("7_chunk").alias("window_type"),
    F.when(
        F.col("sentence_n_minus3").isNotNull() &
        F.col("sentence_n_minus2").isNotNull() &
        F.col("sentence_n_minus1").isNotNull() & 
        F.col("sentence_n").isNotNull() & 
        F.col("sentence_n_plus1").isNotNull() &
        F.col("sentence_n_plus2").isNotNull() &
        F.col("sentence_n_plus3").isNotNull(),
        F.concat_ws(" ",
            F.col("sentence_n_minus3"),
            F.col("sentence_n_minus2"),
            F.col("sentence_n_minus1"),
            F.col("sentence_n"),
            F.col("sentence_n_plus1"),
            F.col("sentence_n_plus2"),
            F.col("sentence_n_plus3")
        )
    ).otherwise(F.lit(None)).alias("window_text"),
    (F.col("sentence_n_minus3").isNotNull() &
     F.col("sentence_n_minus2").isNotNull() &
     F.col("sentence_n_minus1").isNotNull() & 
     F.col("sentence_n").isNotNull() & 
     F.col("sentence_n_plus1").isNotNull() &
     F.col("sentence_n_plus2").isNotNull() &
     F.col("sentence_n_plus3").isNotNull()).alias("has_full_context")
).filter(F.col("sentence_n").isNotNull())

# Combine all windows
all_windows = window_1chunk.union(window_3chunk).union(window_5chunk).union(window_7chunk)

print(f"✅ Window configurations created:")
print(f"   - 1 chunk: {window_1chunk.count()} windows")
print(f"   - 3 chunk: {window_3chunk.count()} windows")
print(f"   - 5 chunk: {window_5chunk.count()} windows")
print(f"   - 7 chunk: {window_7chunk.count()} windows")
print(f"   - Total: {all_windows.count()} windows")

display(all_windows.limit(10))



In [ ]:
# ==================================================
# LLAMA SENTIMENT ANALYSIS (RUNNING FIRST - TO CATCH ERRORS EARLY)
# ==================================================
print("🦙 Running Llama sentiment analysis on all window sizes...")

# Convert windows to list for processing
windows_list = all_windows.collect()
print(f"✅ Collected {len(windows_list)} windows for processing")

LLAMA_PROMPT = (
    'Financial sentiment analysis on Example Company stock performance (EXAMPLE). '
    'Rate whether the given information is bullish and optimistic, neutral, or bearish and pessimistic on EXAMPLE. '
    'Return ONLY strict JSON: {\\"id\\":\\"<id>\\",\\"score\\":<decimal>}. '
    'Scores must be decimal numbers ranging from 0.0 to 7.0. '
    'Scale: 0.0=objective (NON-commentary), 1.0=very pessimistic, 4.0=neutral commentary, 7.0=very optimistic. '
    'Use 0.0 ONLY for objective facts; use 1.0–7.0 for subjective/opinionated commentary. '
    'Text: '
)

def score_llama_with_retry(row, max_retries=3):
    """Score with retry logic and exponential backoff"""
    if row.window_text is None:
        return {
            "window_type": row.window_type,
            "report_id": row.report_id,
            "sentence_number": row.sentence_number,
            "llama_score": None
        }
    
    for attempt in range(max_retries):
        try:
            full_prompt = LLAMA_PROMPT + f"[{row.window_type}|{row.report_id}|{row.sentence_number}]: {row.window_text[:2000]}"
            # Reduce max_tokens and add timeout handling
            response = db_client.predict(
                endpoint=LLAMA,
                inputs={
                    "messages": [{"role": "user", "content": full_prompt}],
                    "temperature": 0.0,
                    "max_tokens": 30  # Reduced from 50 to speed up responses
                }
            )
            # Extract content - matches working version exactly
            raw_content = response["choices"][0]["message"]["content"]
            
            # extract_json always returns a dict (never None) - defaults to {"score": 4} if parsing fails
            # Wrap in try/except as final safety net
            try:
                result = extract_json(raw_content)
            except Exception:
                result = {"score": 4}
            
            # Defensive check - ensure result is always a dict (should never be None, but just in case)
            if result is None or not isinstance(result, dict):
                result = {"score": 4}
            
            # Safely get score with default - multiple layers of protection
            try:
                score = float(result.get("score", 4))
            except (AttributeError, TypeError, ValueError):
                score = 4.0
            
            # Validate score range
            if not (0 <= score <= 7):
                return {
                    "window_type": row.window_type,
                    "report_id": row.report_id,
                    "sentence_number": row.sentence_number,
                    "llama_score": "Error"
                }
            
            return {
                "window_type": row.window_type,
                "report_id": row.report_id,
                "sentence_number": row.sentence_number,
                "llama_score": score
            }
                
        except Exception as e:
            error_str = str(e)
            # If timeout, return default score immediately instead of retrying
            if "timeout" in error_str.lower() or "timed out" in error_str.lower():
                print(f"Llama timeout for [{row.window_type}|{row.report_id}|{row.sentence_number}]: returning default score 4")
                return {
                    "window_type": row.window_type,
                    "report_id": row.report_id,
                    "sentence_number": row.sentence_number,
                    "llama_score": 4  # Return 4 for timeout (neutral score) instead of None
                }
            
            if attempt < max_retries - 1:
                wait_time = (2 ** attempt) + (attempt * 0.5)
                print(f"Llama retry {attempt + 1}/{max_retries} after {wait_time}s: {error_str[:50]}")
                sleep(wait_time)
                continue
            else:
                print(f"Llama failed after {max_retries} attempts: {error_str[:100]}")
                # Return default score 4 on final failure (matches working version)
                return {
                    "window_type": row.window_type,
                    "report_id": row.report_id,
                    "sentence_number": row.sentence_number,
                    "llama_score": 4  # Return 4 instead of None for failed attempts
                }

def process_llama_in_batches(windows_list, batch_size=10, workers=3):
    """Process in batches to avoid overwhelming the endpoint - further reduced for stability"""
    all_results = []
    total = len(windows_list)
    
    for batch_start in range(0, total, batch_size):
        batch_end = min(batch_start + batch_size, total)
        batch = windows_list[batch_start:batch_end]
        
        print(f"Llama: Processing batch {batch_start}-{batch_end}/{total} ({100*batch_start/total:.1f}%)")
        
        with ThreadPoolExecutor(max_workers=workers) as executor:
            futures = [executor.submit(score_llama_with_retry, row) for row in batch]
            for j, future in enumerate(as_completed(futures)):
                if (batch_start + j) % 10 == 0 or (batch_start + j) == total - 1:
                    print(f"Llama: Processed {batch_start + j + 1}/{total} windows ({100*(batch_start + j + 1)/total:.1f}%)")
                all_results.append(future.result())
        
        # Longer pause between batches to avoid overwhelming endpoint
        if batch_end < total:
            sleep(5)  # Increased from 3 to 5 seconds for better stability
    
    return all_results


llama_results = process_llama_in_batches(windows_list, batch_size=10, workers=3)

llama_df = spark.createDataFrame(llama_results)
print(f"✅ Llama analysis complete: {len(llama_results)} results")



In [ ]:
# ==================================================
# FINBERT SENTIMENT ANALYSIS (RUNNING SECOND)
# ==================================================
print("🔍 Loading FinBERT model...")

device = 0 if torch.cuda.is_available() else -1
fin_tok = AutoTokenizer.from_pretrained(FINBERT_MODEL)
fin_mdl = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL)
fin_pipe = pipeline("text-classification", model=fin_mdl, tokenizer=fin_tok,
                    top_k=None, device=device)

print("✅ FinBERT model loaded")

def score_finbert_batch(batch_rows):
    """Score a batch of rows with FinBERT"""
    texts = [r.window_text or "" for r in batch_rows]
    try:
        outs = fin_pipe(texts, truncation=True, max_length=512, batch_size=32)
        results = []
        for r, scores in zip(batch_rows, outs):
            if r.window_text is None:
                score = None
            else:
                probs = {s["label"].lower(): float(s["score"]) for s in scores}
                p_pos = probs.get("positive", 0.0)
                p_neg = probs.get("negative", 0.0)
                score = polarity_to_1_7(p_pos, p_neg)
                # Round to 3 significant figures (e.g., 5.45)
                score = round_to_sig_figs(score, sig_figs=3)
            
            results.append({
                "window_type": r.window_type,
                "report_id": r.report_id,
                "sentence_number": r.sentence_number,
                "finbert_score": score
            })
        return results
    except Exception as e:
        print(f"FinBERT batch error: {e}")
        return [{
            "window_type": r.window_type,
            "report_id": r.report_id,
            "sentence_number": r.sentence_number,
            "finbert_score": "Error" if r.window_text is not None else None
        } for r in batch_rows]

finbert_results = []
batch_rows = []
BATCH_SIZE_FINBERT = 128

print("🔁 Scoring with FinBERT...")
# Collect data first (more efficient than toLocalIterator for large datasets)
print("📥 Collecting window data...")
windows_data = all_windows.select("report_id", "sentence_number", "window_type", "window_text").collect()
total_windows = len(windows_data)
print(f"✅ Collected {total_windows} windows to process")

print("🔁 Scoring with FinBERT...")
for i, row in enumerate(windows_data):
    batch_rows.append(row)
    if len(batch_rows) >= BATCH_SIZE_FINBERT:
        finbert_results.extend(score_finbert_batch(batch_rows))
        batch_rows.clear()
        if len(finbert_results) % 500 == 0:
            print(f"FinBERT: Processed {len(finbert_results)}/{total_windows} windows ({100*len(finbert_results)/total_windows:.1f}%)")

if batch_rows:
    finbert_results.extend(score_finbert_batch(batch_rows))

finbert_df = spark.createDataFrame(finbert_results)
print(f"✅ FinBERT analysis complete: {len(finbert_results)} results")



In [ ]:
# ==================================================
# CLAUDE SENTIMENT ANALYSIS (RUNNING THIRD - AFTER LLAMA AND FINBERT)
# ==================================================
print("🤖 Running Claude sentiment analysis on all window sizes...")
print("Each window size will be rated separately based on its specific context:\n")
print("  - Window 1: Claude sees 1 sentence and rates based on that 1 sentence")
print("  - Window 3: Claude sees 3 sentences and rates based on that 3-sentence context")
print("  - Window 5: Claude sees 5 sentences and rates based on that 5-sentence context")
print("  - Window 7: Claude sees 7 sentences and rates based on that 7-sentence context\n")

CLAUDE_PROMPT = (
    "You are rating Wall Street financial analysis on Example Company stock performance (EXAMPLE). "
    "Rate whether the given information is bullish and optimistic, neutral, or bearish and pessimistic on EXAMPLE. "
    "You might get mentions of Example Company's competitors' performance, but remember to rate on EXAMPLE performance only.\n\n"
    "Return ONLY a valid JSON array, where each item is {{\"id\":\"<id>\",\"score\":<decimal>}}.\n"
    "Scores must be decimal numbers with 3 significant figures (e.g., 2.35, 3.67, 4.51, 5.83, 6.42), ranging from 0.0 to 7.0.\n"
    "Scale: 0.0 = objective (NON-commentary), 1.0 = very pessimistic, 4.0 = neutral commentary, 7.0 = very optimistic.\n"
    "Use 0.0 ONLY for objective facts; use 1.0–7.0 for subjective/opinionated commentary.\n"
    "Use the FULL range with granular decimal precision - do NOT default to whole numbers. Use distinct scores (e.g., 4.23, 4.51, 4.67) rather than repeating the same values.\n\n"
    "Items:\n{items}\n"
    "Respond with ONLY the JSON array."
)

# windows_list already defined in Llama cell (cell 7)
# Define BATCH_SIZE if not already defined
if 'BATCH_SIZE' not in globals():
    BATCH_SIZE = 5

def process_claude_batch(batch):
    valid_batch = [row for row in batch if row.window_text is not None]
    
    if not valid_batch:
        return [{
            "window_type": row.window_type,
            "report_id": row.report_id,
            "sentence_number": row.sentence_number,
            "claude_score": None
        } for row in batch]
    
    error_keys = set()
    items_list = []
    for row in valid_batch:
        truncated_text = row.window_text[:1500] if row.window_text else ""
        estimated_tokens = len(truncated_text) / 4
        key = (row.window_type, row.report_id, row.sentence_number)
        if estimated_tokens > 8000:
            error_keys.add(key)
        else:
            items_list.append((row, truncated_text))
    
    items_str = "\n".join([f"[{row.window_type}|{row.report_id}|{row.sentence_number}]: {text}" 
                          for row, text in items_list])
    
    if not items_str and valid_batch:
        return [{
            "window_type": row.window_type,
            "report_id": row.report_id,
            "sentence_number": row.sentence_number,
            "claude_score": "Error" if row.window_text is not None else None
        } for row in batch]
    
    try:
        response = claude_client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=500,
            messages=[{"role": "user", "content": CLAUDE_PROMPT.format(items=items_str)}]
        )
        
        response_text = response.content[0].text.strip()
        
        if "token" in response_text.lower() and ("limit" in response_text.lower() or "exceeded" in response_text.lower()):
            return [{
                "window_type": row.window_type,
                "report_id": row.report_id,
                "sentence_number": row.sentence_number,
                "claude_score": "Error" if row.window_text is not None else None
            } for row in batch]
        
        try:
            results = json.loads(response_text)
        except json.JSONDecodeError:
            json_match = re.search(r'\[.*\]', response_text, re.DOTALL)
            if json_match:
                try:
                    results = json.loads(json_match.group())
                except:
                    return [{
                        "window_type": row.window_type,
                        "report_id": row.report_id,
                        "sentence_number": row.sentence_number,
                        "claude_score": "Error" if row.window_text is not None else None
                    } for row in batch]
            else:
                return [{
                    "window_type": row.window_type,
                    "report_id": row.report_id,
                    "sentence_number": row.sentence_number,
                    "claude_score": "Error" if row.window_text is not None else None
                } for row in batch]
        
        if not isinstance(results, list):
            return [{
                "window_type": row.window_type,
                "report_id": row.report_id,
                "sentence_number": row.sentence_number,
                "claude_score": "Error" if row.window_text is not None else None
            } for row in batch]
        
        output = []
        for item in results:
            try:
                if "id" not in item or "score" not in item:
                    continue
                
                item_id = item["id"].strip("[]")
                parts = item_id.split("|")
                
                if len(parts) != 3:
                    continue
                
                report_id = int(parts[1])
                sentence_number = int(parts[2])
                score = float(item["score"])
                
                if not (0 <= score <= 7):
                    output.append({
                        "window_type": parts[0],
                        "report_id": report_id,
                        "sentence_number": sentence_number,
                        "claude_score": "Error"
                    })
                else:
                    output.append({
                        "window_type": parts[0],
                        "report_id": report_id,
                        "sentence_number": sentence_number,
                        "claude_score": score
                    })
            except (ValueError, IndexError):
                continue
        
        score_map = {(item["window_type"], item["report_id"], item["sentence_number"]): item["claude_score"] 
                     for item in output}
        
        final_output = []
        for row in batch:
            key = (row.window_type, row.report_id, row.sentence_number)
            if row.window_text is None:
                score = None
            elif key in error_keys:
                score = "Error"
            elif key in score_map:
                score = score_map[key]
            else:
                score = "Error"
            
            final_output.append({
                "window_type": row.window_type,
                "report_id": row.report_id,
                "sentence_number": row.sentence_number,
                "claude_score": score
            })
        
        return final_output
        
    except Exception as e:
        error_msg = str(e).lower()
        is_token_error = "token" in error_msg and ("limit" in error_msg or "exceeded" in error_msg)
        is_timeout = "timeout" in error_msg or "timed out" in error_msg
        is_api_error = "api" in error_msg or "rate limit" in error_msg or "429" in error_msg
        
        if is_token_error or is_timeout or is_api_error:
            return [{
                "window_type": row.window_type,
                "report_id": row.report_id,
                "sentence_number": row.sentence_number,
                "claude_score": "Error" if row.window_text is not None else None
            } for row in batch]
        
        return [{
            "window_type": row.window_type,
            "report_id": row.report_id,
            "sentence_number": row.sentence_number,
            "claude_score": "Error" if row.window_text is not None else None
        } for row in batch]

claude_batches = [windows_list[i:i + BATCH_SIZE] for i in range(0, len(windows_list), BATCH_SIZE)]
claude_results = []

print(f"Processing {len(claude_batches)} batches...")
with ThreadPoolExecutor(max_workers=15) as executor:
    futures = {executor.submit(process_claude_batch, batch): batch for batch in claude_batches}
    
    for i, future in enumerate(as_completed(futures)):
        if i % 20 == 0:
            print(f"Claude: Processed {i}/{len(claude_batches)} batches")
        claude_results.extend(future.result())

claude_df = spark.createDataFrame(claude_results)
print(f"✅ Claude analysis complete: {len(claude_results)} results")



In [ ]:
# ==================================================
# PIVOT RESULTS BY WINDOW TYPE FOR EACH MODEL
# ==================================================
print("🔄 Pivoting results by window type...")

# Pivot Claude results
claude_pivoted = (claude_df
    .groupBy("report_id", "sentence_number")
    .pivot("window_type", ["1_chunk", "3_chunk", "5_chunk", "7_chunk"])
    .agg(F.first("claude_score"))
    .withColumnRenamed("1_chunk", "claude_score_1chunk")
    .withColumnRenamed("3_chunk", "claude_score_3chunk")
    .withColumnRenamed("5_chunk", "claude_score_5chunk")
    .withColumnRenamed("7_chunk", "claude_score_7chunk")
)

# Pivot FinBERT results
finbert_pivoted = (finbert_df
    .groupBy("report_id", "sentence_number")
    .pivot("window_type", ["1_chunk", "3_chunk", "5_chunk", "7_chunk"])
    .agg(F.first("finbert_score"))
    .withColumnRenamed("1_chunk", "finbert_score_1chunk")
    .withColumnRenamed("3_chunk", "finbert_score_3chunk")
    .withColumnRenamed("5_chunk", "finbert_score_5chunk")
    .withColumnRenamed("7_chunk", "finbert_score_7chunk")
)

# Pivot Llama results
llama_pivoted = (llama_df
    .groupBy("report_id", "sentence_number")
    .pivot("window_type", ["1_chunk", "3_chunk", "5_chunk", "7_chunk"])
    .agg(F.first("llama_score"))
    .withColumnRenamed("1_chunk", "llama_score_1chunk")
    .withColumnRenamed("3_chunk", "llama_score_3chunk")
    .withColumnRenamed("5_chunk", "llama_score_5chunk")
    .withColumnRenamed("7_chunk", "llama_score_7chunk")
)

print("✅ Results pivoted for all models")



In [ ]:
# ==================================================
# CREATE COMPREHENSIVE COMPARISON SHEET
# ==================================================
print("🎯 Creating comprehensive comparison sheet...")

original_df = (
    df.withColumn("report_id", F.col("`Doc #`").cast("long"))
      .withColumn("sentence_number", F.col("`Sentence #`").cast("long"))
)

# Join all results
comparison_sheet = (
    original_df
    .join(claude_pivoted, ["report_id", "sentence_number"], "left")
    .join(finbert_pivoted, ["report_id", "sentence_number"], "left")
    .join(llama_pivoted, ["report_id", "sentence_number"], "left")
    .select(
        F.col("`Doc #`").alias("doc_num"),
        F.col("`Sentence #`").alias("sentence_num"),
        F.col("`Firm name`").alias("firm_name"),
        F.col("Context").alias("context"),
        # Claude scores
        F.expr("try_cast(claude_score_1chunk as double)").alias("claude_1chunk"),
        F.expr("try_cast(claude_score_3chunk as double)").alias("claude_3chunk"),
        F.expr("try_cast(claude_score_5chunk as double)").alias("claude_5chunk"),
        F.expr("try_cast(claude_score_7chunk as double)").alias("claude_7chunk"),
        # FinBERT scores
        F.expr("try_cast(finbert_score_1chunk as double)").alias("finbert_1chunk"),
        F.expr("try_cast(finbert_score_3chunk as double)").alias("finbert_3chunk"),
        F.expr("try_cast(finbert_score_5chunk as double)").alias("finbert_5chunk"),
        F.expr("try_cast(finbert_score_7chunk as double)").alias("finbert_7chunk"),
        # Llama scores
        F.expr("try_cast(llama_score_1chunk as double)").alias("llama_1chunk"),
        F.expr("try_cast(llama_score_3chunk as double)").alias("llama_3chunk"),
        F.expr("try_cast(llama_score_5chunk as double)").alias("llama_5chunk"),
        F.expr("try_cast(llama_score_7chunk as double)").alias("llama_7chunk")
    )
    .orderBy("doc_num", "sentence_num")
)

print("✅ Comparison sheet created")
print("Note: Scores are raw values from models.")
print("  - NA (null): Missing context at boundaries")
print("  - 0.0: Objective (non-commentary) facts as determined by model")
print("  - 1.0-7.0: Commentary scores (1.0 = very pessimistic, 4.0 = neutral, 7.0 = very optimistic)")
print("  - Error: Invalid responses (timeout, parsing errors, API errors, missing responses, etc.)")
print(f"\n📊 Total rows processed: {comparison_sheet.count()} (showing first 20 rows below)")
display(comparison_sheet.limit(20))



In [ ]:
# ==================================================
# EXPORT RESULTS
# ==================================================
print("💾 Saving results to table...")

comparison_sheet.write.mode("overwrite").saveAsTable("three_models_window_size_comparison")

print("✅ Results saved to table 'three_models_window_size_comparison'")
print("\n🎉 Window size testing complete across all three models!")
print("\nModels tested:")
print("  - Claude (Anthropic)")
print("  - FinBERT (Financial BERT)")
print("  - Llama (Meta)")
print("\nWindow sizes tested:")
print("  - 1 chunk: [n]")
print("  - 3 chunks: [n-1, n, n+1]")
print("  - 5 chunks: [n-2, n-1, n, n+1, n+2]")
print("  - 7 chunks: [n-3, n-2, n-1, n, n+1, n+2, n+3]")

